# Chapter 12 — One Operation, Several Model APIs

**Companion to *Applied AI*.**

This notebook accompanies Chapter 12. On a real gateway, choosing a model
chooses its wire dialect — so every chamber swap is also a protocol swap.

The chapter's central claim is testable and was tested: **the dialect must not
reach the runtime's decision.** That experiment is preserved here.

## Question

**Does the wire dialect change what the runtime decides — and what happens to
each control on its way to three different APIs?**

## What this notebook establishes

- The preserved offline conformance run, read back: nine cases through three
  dialects, and for every case **all three dialects produce an identical
  outcome** for transport, generation, error kind and call status.
- The six fates of a control, implemented, including the two that matter:
  **refusal before any effect**, and **recorded omission**.
- Why `prepare()` once and use the result twice is the fix for a real defect:
  a request body and its record drifting apart.

## What this notebook does **not** establish

- The offline run used a patched transport with **outbound sockets refused**.
  It establishes that the boundary holds, not that any provider behaves well.
- The three live calls in the same bundle used **three different models**, so
  they cannot isolate a protocol effect, and the chapter refuses to read them
  that way. No winner was computed and none should be.
- The six-fates implementation below is a **teaching codec**, not CodeAI's.

## Setup

In [1]:
import json
import os
from collections import defaultdict
from pathlib import Path

def find_evidence_dir(marker="protocol-conformance"):
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError("Set APPLIED_AI_EVIDENCE to the evidence directory.")

EVIDENCE_DIR = find_evidence_dir()
BUNDLE = EVIDENCE_DIR / "protocol-conformance"
report = json.loads((BUNDLE / "offline" / "conformance-report.json").read_text(encoding="utf-8"))

print("EVIDENCE_DIR :", EVIDENCE_DIR)
print("network calls made by the run :", report["network_calls"])
print("cases passed                  :", report["passed"])
print("rows                          :", len(report["rows"]))
print("captured (replayed real) cases:", report["captured_cases"])

EVIDENCE_DIR : $APPLIED_AI_EVIDENCE
network calls made by the run : 0
cases passed                  : 27
rows                          : 27
captured (replayed real) cases: 3


## 1. On this gateway, a model chooses a protocol

The three routes the chapter uses, read from the preserved rows.

In [2]:
routes = {}
for r in report["rows"]:
    routes.setdefault(r["protocol"], set()).add(r["model"])

for proto, models in sorted(routes.items()):
    print(f"  {proto:<20} {sorted(models)}")

print()
print("Choosing the model chose the endpoint, the request shape, the headers,")
print("the fate of at least one control, the completion vocabulary, where")
print("reasoning comes back, and the usage vocabulary.")

  chat_completions     ['mimo-v2.5']
  messages             ['minimax-m2.7']
  responses            ['gpt-5.6-luna']

Choosing the model chose the endpoint, the request shape, the headers,
the fate of at least one control, the completion vocabulary, where
reasoning comes back, and the usage vocabulary.


## 2. The claim: the dialect does not reach the decision

For each case, group the three dialects and compare the outcomes.

In [3]:
def outcome(r):
    return (r["observation"].get("transport_outcome"),
            r["interpretation"].get("generation_state"),
            r["interpretation"].get("error_kind"),
            r["call_status"])

by_case = defaultdict(dict)
for r in report["rows"]:
    by_case[r["case"]][r["protocol"]] = outcome(r)

protos = sorted(routes)
print(f"{'case':<16}{'transport':<20}{'generation':<13}{'error kind':<21}"
      f"{'call status':<13}{'identical?'}")
print("-" * 100)

all_identical = True
for case in sorted(by_case):
    seen = by_case[case]
    vals = set(seen.values())
    same = len(vals) == 1
    all_identical &= same
    t, g, e, s = next(iter(vals)) if same else ("varies",) * 4
    print(f"{case:<16}{str(t):<20}{str(g):<13}{str(e):<21}{str(s):<13}"
          f"{'yes' if same else 'NO'}")

print()
assert all_identical, "a dialect leaked into the decision"
print("assertion held: for every case, all three dialects produced the same")
print("transport, generation, error kind and call status.")

case            transport           generation   error kind           call status  identical?
----------------------------------------------------------------------------------------------------
captured        response_received   complete     None                 succeeded    yes
complete        response_received   complete     None                 succeeded    yes
empty           response_received   empty        empty_output         failed       yes
http_error      http_error          unknown      invalid_request      failed       yes
malformed       response_received   unknown      malformed_response   failed       yes
no_response     no_response         unknown      timeout              failed       yes
tool_only       response_received   empty        empty_output         failed       yes
truncated       response_received   truncated    None                 unresolved   yes
unknown_reason  response_received   unknown      None                 succeeded    yes

assertion held: for e

## Observation

That identity **is** the claim. Transport, generation and call status are
decided from the canonical interpretation, whatever shape the bytes had.

Two rows repay a second look.

In [4]:
for case in ("truncated", "tool_only", "unknown_reason"):
    t, g, e, s = next(iter(set(by_case[case].values())))
    print(f"{case:<16} generation={str(g):<12} call_status={s}")
print()
print("For reference, the full canonical vocabulary observed in this run:")
print("  transport :", sorted({o[0] for v in by_case.values() for o in v.values()}))
print("  generation:", sorted({o[1] for v in by_case.values() for o in v.values()}))
print("  statuses  :", sorted({o[3] for v in by_case.values() for o in v.values()}))

print()
print("truncated      -> the call is UNRESOLVED, not succeeded. A finished")
print("                  transport is not a finished generation.")
print("tool_only      -> a reply with a tool call and no text reads as")
print("                  'empty' and fails. Defensible for a text chamber;")
print("                  wrong for a chamber whose occupant answers that way.")
print("unknown_reason -> still SUCCEEDS. The chapter names this as a weakness:")
print("                  an unrecognised reason is the likeliest shape of a")
print("                  future API change, and it is treated as fine.")

truncated        generation=truncated    call_status=unresolved
tool_only        generation=empty        call_status=failed
unknown_reason   generation=unknown      call_status=succeeded

For reference, the full canonical vocabulary observed in this run:
  transport : ['http_error', 'no_response', 'response_received']
  generation: ['complete', 'empty', 'truncated', 'unknown']
  statuses  : ['failed', 'succeeded', 'unresolved']

truncated      -> the call is UNRESOLVED, not succeeded. A finished
                  transport is not a finished generation.
tool_only      -> a reply with a tool call and no text reads as
                  'empty' and fails. Defensible for a text chamber;
                  wrong for a chamber whose occupant answers that way.
unknown_reason -> still SUCCEEDS. The chapter names this as a weakness:
                  an unrecognised reason is the likeliest shape of a
                  future API change, and it is treated as fine.


## 3. Replaying the real captures

Three of the twenty-seven rows are not synthetic: they replay response bytes
actually captured from the gateway.

In [5]:
captured = [r for r in report["rows"] if r["case"] == "captured"]
print(f"{'protocol':<20}{'model':<18}{'generation':<12}{'status':<12}{'answer (truncated)'}")
print("-" * 104)
for r in sorted(captured, key=lambda x: x["protocol"]):
    txt = (r.get("canonical_text") or "").replace("\n", " ")[:34]
    model = str(r.get('model') or '-')
    gen = str(r['interpretation'].get('generation_state') or '-')
    print(f"{r['protocol']:<20}{model:<18}{gen:<12}"
          f"{r['call_status']:<12}{txt}...")

print()
print("The canonical text holds ONLY the answer. Reasoning and tool blocks stay")
print("in the preserved response bytes - merging them would make the answer")
print("wrong, and deleting them would destroy an observation.")

protocol            model             generation  status      answer (truncated)
--------------------------------------------------------------------------------------------------------
chat_completions    mimo-v2.5         complete    succeeded   The claim lacks empirical performa...
messages            minimax-m2.7      complete    succeeded   No measurement data or benchmark r...
responses           gpt-5.6-luna      complete    succeeded   No benchmark or profiling data is ...

The canonical text holds ONLY the answer. Reasoning and tool blocks stay
in the preserved response bytes - merging them would make the answer
wrong, and deleting them would destroy an observation.


## 4. The six fates of a control

Now the mechanism, implemented small. Every control a caller asks for meets one
of six fates on its way to a dialect.

In [6]:
class UnknownControlError(Exception): pass
class InvalidControlError(Exception): pass

KNOWN = {"max_tokens", "temperature", "reasoning_effort", "seed"}

SPECS = {
    "responses": {
        "max_tokens":       ("rename", "max_output_tokens"),
        "temperature":      ("as_is",  "temperature"),
        "reasoning_effort": ("nest",   ("reasoning", "effort")),
        "seed":             ("omit",   None),
    },
    "chat_completions": {
        "max_tokens":       ("as_is",  "max_tokens"),
        "temperature":      ("as_is",  "temperature"),
        "reasoning_effort": ("omit",   None),
        "seed":             ("omit",   None),
    },
    "messages": {
        "max_tokens":       ("as_is",  "max_tokens"),
        "temperature":      ("as_is",  "temperature"),
        "reasoning_effort": ("omit",   None),
        "seed":             ("omit",   None),
    },
}
REQUIRES_LIMIT = {"messages"}          # Messages requires an output limit
DEFAULT_LIMIT = 1024

def prepare(protocol, requested: dict):
    """Validate, map, default, refuse - all BEFORE any request exists."""
    body, omitted, defaulted = {}, [], {}

    for name, value in requested.items():
        if name not in KNOWN:
            raise UnknownControlError(
                f"unknown control {name!r}: refused before any effect")
        if name in ("max_tokens",) and not isinstance(value, int):
            raise InvalidControlError(
                f"{name}={value!r} is not an integer: refused before any effect")

        how, target = SPECS[protocol][name]
        if how == "as_is":
            body[target] = value
        elif how == "rename":
            body[target] = value
        elif how == "nest":
            outer, inner = target
            body.setdefault(outer, {})[inner] = value
        elif how == "omit":
            omitted.append(name)

    limit_key = SPECS[protocol]["max_tokens"][1]
    if protocol in REQUIRES_LIMIT and limit_key not in body:
        body[limit_key] = DEFAULT_LIMIT
        defaulted[limit_key] = DEFAULT_LIMIT

    return {"protocol": protocol, "body": body,
            "requested": dict(requested),
            "omitted_unsupported": omitted,
            "defaulted_controls": defaulted}

In [7]:
requested = {"max_tokens": 256, "temperature": 0.2, "reasoning_effort": "low"}

print("requested   ", json.dumps(requested))
print()
for proto in ("responses", "chat_completions", "messages"):
    p = prepare(proto, requested)
    line = f"{proto:<18} sent {json.dumps(p['body'])}"
    if p["omitted_unsupported"]:
        line += f"   omitted_unsupported: {p['omitted_unsupported']}"
    print(line)

print()
p = prepare("messages", {"temperature": 0.2})
print("messages with NO limit requested:")
print("   sent      ", json.dumps(p["body"]))
print("   defaulted ", json.dumps(p["defaulted_controls"]))

requested    {"max_tokens": 256, "temperature": 0.2, "reasoning_effort": "low"}

responses          sent {"max_output_tokens": 256, "temperature": 0.2, "reasoning": {"effort": "low"}}
chat_completions   sent {"max_tokens": 256, "temperature": 0.2}   omitted_unsupported: ['reasoning_effort']
messages           sent {"max_tokens": 256, "temperature": 0.2}   omitted_unsupported: ['reasoning_effort']

messages with NO limit requested:
   sent       {"temperature": 0.2, "max_tokens": 1024}
   defaulted  {"max_tokens": 1024}


Fates five and six are where systems quietly lie.

> **A default nobody records is a control the caller never chose and cannot
> see.** If one route silently supplies an output limit while another sends
> none, any comparison between those chambers carries a hidden variable.

## 5. Refusal before any effect

The chapter's two historical defects, and what the corrected boundary does.

In [8]:
class Transport:
    def __init__(self): self.calls = 0
    def send(self, prepared):
        self.calls += 1
        return {"ok": True}

for label, payload in [
    ("malformed value  (max_tokens='abc')", {"max_tokens": "abc"}),
    ("unknown control  (top_k2=4)",         {"top_k2": 4}),
    ("valid request",                       {"max_tokens": 256}),
]:
    t = Transport()
    try:
        prep = prepare("chat_completions", payload)
        t.send(prep)
        print(f"{label:<38} sent, transport calls = {t.calls}")
    except (UnknownControlError, InvalidControlError) as exc:
        print(f"{label:<38} REFUSED: {exc}")
        print(f"{'':<38} transport calls = {t.calls}")
        assert t.calls == 0

print()
print("assertion held: a refused control produces no request at all.")
print()
print("The chapter's earlier behaviour: a malformed max_tokens was dropped from")
print("the body and still recorded as sent; reasoning_effort was accepted,")
print("never sent, and left no trace. Both are silent-configuration debt: every")
print("comparison built on the record inherits the error.")

malformed value  (max_tokens='abc')    REFUSED: max_tokens='abc' is not an integer: refused before any effect
                                       transport calls = 0
unknown control  (top_k2=4)            REFUSED: unknown control 'top_k2': refused before any effect
                                       transport calls = 0
valid request                          sent, transport calls = 1

assertion held: a refused control produces no request at all.

The chapter's earlier behaviour: a malformed max_tokens was dropped from
the body and still recorded as sent; reasoning_effort was accepted,
never sent, and left no trace. Both are silent-configuration debt: every
comparison built on the record inherits the error.


## 6. Send what you record

The structural cause of that first defect was two code paths producing two
views of one request. The fix is to prepare once and use the result twice.

In [9]:
import hashlib

def semantic_hash(prepared):
    return hashlib.sha256(
        json.dumps(prepared["body"], sort_keys=True).encode()).hexdigest()

prep = prepare("responses", requested)
manifest = {"effective": prep["body"], "hash": semantic_hash(prep)}   # recorded
sent_1 = prep["body"]                                                 # transmitted
sent_2 = prep["body"]                                                 # retried

assert manifest["effective"] == sent_1 == sent_2
assert semantic_hash({"body": sent_1}) == manifest["hash"]
print("assertion held: the recorded body, the sent body and the retried body")
print("are the same object.")
print()
print("request hash:", manifest["hash"][:24], "...")
print()
print("Two precise limits from the chapter:")
print("  - the hash identifies the SEMANTIC request, not the outbound bytes")
print("  - credentials never enter the prepared object; they are applied")
print("    inside send()")

assertion held: the recorded body, the sent body and the retried body
are the same object.

request hash: 4483980184d6e1b711340377 ...

Two precise limits from the chapter:
  - the hash identifies the SEMANTIC request, not the outbound bytes
  - credentials never enter the prepared object; they are applied
    inside send()


## Interpretation

The chapter's architecture, confirmed by the preserved run:

- **Dialect knowledge lives in the adapter.** CodeAI's runtime contains no
  protocol branch, which is Parnas's criterion applied to the part of the
  system most likely to change — on this gateway, every time an occupant does.
- **Contain differences; do not hide them.** Translation is fine. Silent
  tolerance is debt, and RFC 9413's case against liberal acceptance shows up
  directly in CodeAI's own history.
- **Success has layers.** Transport, completion, answer presence, syntax,
  shape, values, task criteria and verification each answer a different
  question. The `truncated` row above is the cheapest possible demonstration:
  HTTP succeeded and the generation did not.

And the weakness the run records rather than hides: an **unknown completion
reason still produces a succeeded call**, which is the likeliest shape of a
future API change.

## Try it yourself

1. **Add a fourth dialect** to `SPECS` that renames `temperature`. Re-run the
   six-fates cell. How much of the runtime had to change? (None of it — that is
   the point.)
2. **Make `unknown_reason` fail.** Change the mapping so an unrecognised finish
   reason produces `unresolved` rather than `succeeded`, then argue the case
   both ways. Which chambers would that break?
3. **Remove the recorded omission.** Drop `omitted_unsupported` and compare two
   chambers whose `reasoning_effort` silently vanished on one side. What would
   you have concluded from the comparison?
4. **Inspect a captured row in full.** Print one `captured` row's `manifest` and
   `usage` and compare the three routes' usage vocabularies. Chapter 13 is about
   why those numbers do not add up.